# SwinUNETR spleen segmentation: train → ONNX → TensorRT FP16 benchmark

This notebook reproduces every number in the project README on a free Google Colab **T4 GPU**.

**Before you start:** `Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save`.

What it does, in order:

1. Checks that a GPU is attached
2. Clones the repository and installs pinned dependencies (+ TensorRT)
3. Runs the unit tests
4. Downloads the MSD Task09 Spleen dataset (~1.5 GB)
5. Trains SwinUNETR (the longest step; see the note in that section)
6. Exports the best checkpoint to ONNX and verifies it against PyTorch
7. Builds a TensorRT FP16 engine and benchmarks PyTorch FP32 vs PyTorch AMP vs TensorRT FP16
8. Plots the training curve and saves everything to `results/`

Every step is a plain `python -m src.<script>` call, so you can run the same commands on any CUDA machine.
See `docs/COLAB.md` in the repo for a step-by-step guide with troubleshooting.

## 1. Check the GPU
You should see a **Tesla T4** below. If you see an error, the runtime has no GPU: change the runtime type and re-run.

In [ ]:
!nvidia-smi

## 2. (Optional but recommended) Save checkpoints to Google Drive

Colab can disconnect after idle time or after ~12 hours, and everything on the VM is then deleted.
Running this cell stores checkpoints and results on your Drive so a disconnect does not lose your trained model.
Skip it if you don't mind re-training.

In [ ]:
USE_DRIVE = True  # set to False to keep everything on the (temporary) Colab VM

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/swinunetr-spleen-tensorrt'
    !mkdir -p "{DRIVE_DIR}/checkpoints" "{DRIVE_DIR}/results"

## 3. Clone the repository and install dependencies
Colab already has a CUDA build of PyTorch, so `requirements-colab.txt` deliberately leaves `torch` out. TensorRT 10 is installed from NVIDIA's pip wheel for CUDA 12 (pinned in that file).

In [ ]:
%cd /content
!rm -rf swinunetr-spleen-tensorrt
!git clone https://github.com/kshitijv97/swinunetr-spleen-tensorrt.git
%cd /content/swinunetr-spleen-tensorrt
!pip install -q -r requirements-colab.txt   # includes tensorrt-cu12==10.16.1.11

In [ ]:
# If you mounted Drive, make checkpoints/ and results/ point at it (symlinks).
if USE_DRIVE:
    !rm -rf checkpoints && ln -s "{DRIVE_DIR}/checkpoints" checkpoints
    !cp -n results/.gitkeep "{DRIVE_DIR}/results/" 2>/dev/null; rm -rf results && ln -s "{DRIVE_DIR}/results" results
!ls -la

In [ ]:
import torch, monai, tensorrt
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU', torch.cuda.get_device_name(0))
print('monai', monai.__version__, '| tensorrt', tensorrt.__version__)

## 4. Run the unit tests
Fast sanity checks (synthetic data, tiny model). All should pass.

In [ ]:
!python -m pytest -q

## 5. Download the dataset
MSD Task09 Spleen: 41 labelled CT volumes. The download is checksum-verified. ~1-3 minutes.

In [ ]:
!python -m src.data --download --summary

## 6. Train

Default settings live in `configs/default.yaml` (feature size 48, 96³ crops, 150 epochs, validation every 5 epochs, mixed precision).
The best checkpoint (by validation Dice) is saved to `checkpoints/best.pt`.

**Time:** this is the long step. Each epoch's time is printed, so after a few epochs you can estimate the total
(epoch seconds × 150, plus validation). If it is too long for your session, lower `EPOCHS` below; the README should then say how many epochs you used.

If Colab runs out of **system RAM** while caching the dataset, add `--max-volumes 30` or edit `data.cache_rate` in the config.
If the **GPU** runs out of memory, set `model.use_checkpoint: true` in the config.

In [ ]:
EPOCHS = 150  # lower this if your Colab session is short
!python -m src.train --epochs {EPOCHS}

## 7. Export to ONNX and verify
Exports with a fixed input shape `(4, 1, 96, 96, 96)` and checks ONNX Runtime matches PyTorch within tolerance.

In [ ]:
!python -m src.export_onnx

## 8. Benchmark PyTorch FP32 vs PyTorch AMP vs TensorRT FP16

The first run builds the TensorRT engine (several minutes; TensorRT times many kernel candidates).
Latency is per **whole validation volume** with sliding-window inference, after warm-up, with CUDA synchronisation.
See `docs/BENCHMARKING.md` for exactly what is and isn't measured.

In [ ]:
!python -m src.benchmark

## 9. Plot the training curve

In [ ]:
import json
import matplotlib.pyplot as plt

h = json.load(open('results/training_history.json'))
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(h['epoch'], h['train_loss'], label='train loss (DiceCE)', color='tab:blue')
ax1.set_xlabel('epoch'); ax1.set_ylabel('train loss')
ax2 = ax1.twinx()
ax2.plot(h['val_epoch'], h['val_dice'], 'o-', label='val Dice', color='tab:orange')
ax2.set_ylabel('validation Dice'); ax2.set_ylim(0, 1)
fig.legend(loc='center right'); fig.tight_layout()
fig.savefig('results/training_curve.png', dpi=150)
print(f"best val Dice {h['best_val_dice']:.4f} at epoch {h['best_epoch']}, training took {h['train_minutes']:.1f} min")

## 10. The numbers to copy into the README
Copy the table below into the **Results** section of `README.md`. `docs/REPORTING_RESULTS.md` explains each number.

In [ ]:
print(open('results/benchmark.md').read())
b = json.load(open('results/benchmark.json'))
e = json.load(open('results/onnx_export.json'))
print('ONNX max abs diff vs PyTorch:', f"{e['max_abs_diff']:.2e}", '| argmax agreement:', f"{e['argmax_agreement']*100:.4f}%")
print('TensorRT engine:', b['tensorrt_engine'])
print('Environment:', b['environment'])

## 11. Download the results
Downloads a zip of `results/` (small JSON/Markdown/PNG files - no data or weights). Unzip it into your local clone, commit, and push.

In [ ]:
!cd /content/swinunetr-spleen-tensorrt && zip -r -j /content/results.zip results/*.json results/*.md results/*.png
from google.colab import files
files.download('/content/results.zip')